# Kaggle runner - one notebook for every GPU job

**What to do:** (1) set `SESSION` in cell 1, (2) check Settings: **GPU T4 x2** and **Internet ON**,
(3) click **Save Version -> Save & Run All** and let it run in the background (you can close the browser),
(4) when it is finished, download `results_session<N>.zip` from the *Output* tab and upload it to the chat.

| SESSION | Jobs | Approx. time |
|---|---|---|
| 1 | ALLaM config 2, beam-search re-run of it, ALLaM config 3, ALLaM config 1 | 3.5 h |
| 2 | AraT5v2 final run, Mistral config 2, Mistral config 1 | 2.5 h |
| 3 | LLaMA 3 configs 2, 4, 1 (needs the HF_TOKEN secret and the Llama 3 licence) | 3.5 h |
| 4 | Zero-shot ALLaM, Mistral, LLaMA 3 (needs HF_TOKEN) | 3 h |
| 5 | Neural metrics COMET / COMETKiwi / BLEURT for ALL runs (only after the results were pushed to GitHub) | 0.5 h |

Use `DRY_RUN = True` to test a session in a few minutes before the real run.

## 1. Settings (the only cell you edit)

In [ ]:
SESSION = 1        # 1, 2, 3, 4 or 5  (see the table above)
DRY_RUN = False    # True = tiny smoke test of every job (a few minutes)

## 2. Get the code from GitHub

In [ ]:
import os, glob, subprocess

REPO_URL = "https://github.com/asmabenhenia/Reproduction-of-domain-specific-Tunisian-Arabic-dialect-to-MSA-translation.git"

WORK = "/kaggle/working"
name = REPO_URL.rstrip("/").removesuffix(".git").split("/")[-1]
path = os.path.join(WORK, name)
if not os.path.exists(path):
    subprocess.run(["git", "clone", REPO_URL, path], check=True)
else:
    subprocess.run(["git", "-C", path, "checkout", "--", "."], check=True)
    subprocess.run(["git", "-C", path, "pull"], check=True)
hits = sorted(glob.glob(path + "/**/src/train_lora.py", recursive=True), key=len)
assert hits, "src/train_lora.py not found: check REPO_URL and that you pushed the files"
os.chdir(os.path.dirname(os.path.dirname(hits[0])))
print("working directory:", os.getcwd())
assert os.path.exists("src/neural_eval.py"), "Old code on GitHub: unzip the newest tun_msa_repro.zip into your repo, then git add . / commit / push, and re-run this cell"
print("code is up to date")

## 3. Install libraries

In [ ]:
get_ipython().system('pip install -q "transformers>=4.46,<5" "peft>=0.12" bitsandbytes accelerate sacrebleu pyyaml sentencepiece protobuf scikit-learn openpyxl')
if SESSION == 5:   # neural metrics need extra packages (they may change library versions, hence their own session)
    get_ipython().system('pip install -q unbabel-comet "git+https://github.com/lucadiliello/bleurt-pytorch.git"')

## 4. Hugging Face token (needed for LLaMA 3 and COMETKiwi; harmless otherwise)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret (fine for ALLaM / Mistral / AraT5v2):", type(e).__name__)

## 5. Run the jobs of this session

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_lora.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 2 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

import subprocess

PRESETS = {
    1: ["lora allam7b_c2", "beam allam7b_c2", "lora allam7b_c3", "lora allam7b_c1"],
    2: ["seq2seq arat5v2", "lora mistral7b_c2", "lora mistral7b_c1"],
    3: ["lora llama3_8b_c2", "lora llama3_8b_c4", "lora llama3_8b_c1"],
    4: ["zeroshot allam7b", "zeroshot mistral7b", "zeroshot llama3_8b"],
    5: ["neural all"],
}

def to_cmd(job):
    kind, _, arg = job.partition(" ")
    dry = " --dry_run" if DRY_RUN else ""
    if kind == "lora":
        return f"python -m src.train_lora --config configs/lora_{arg}.yaml{dry}"
    if kind == "seq2seq":
        return f"python -m src.train_seq2seq --config configs/seq2seq_{arg}.yaml --tag final{dry}"
    if kind == "zeroshot":
        return f"python -m src.generate zeroshot --model {arg}{dry}"
    if kind == "beam":
        adapter = (f"results/_dryrun/lora_{arg}_seed42_dryrun/adapter" if DRY_RUN
                   else f"results/table8_lora/lora_{arg}_seed42/adapter")
        if not os.path.isdir(adapter):
            return None
        return f"python -m src.generate beam --config configs/lora_{arg}.yaml --adapter {adapter} --beams 4{dry}"
    if kind == "neural":
        return "python -m src.neural_eval --all"
    raise ValueError(job)

def run(cmd):
    """Run a command and stream its output into the notebook (progress bars collapsed)."""
    print("\n$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        seg = [x for x in line.split("\r") if x.strip()]
        if seg:
            print(seg[-1], end="" if seg[-1].endswith("\n") else "\n", flush=True)
    return p.wait()

status = {}
for job in PRESETS[SESSION]:
    cmd = to_cmd(job)
    if cmd is None:
        status[job] = "SKIPPED (adapter not found: its LoRA run must finish first)"
        continue
    rc = run(cmd)
    status[job] = "OK" if rc == 0 else f"FAILED (exit code {rc})"
print("\n=== SESSION", SESSION, "STATUS ===")
for k, v in status.items():
    print(f"{v:<60} {k}")

## 6. Summary table - send me this output as text

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_lora.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 2 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

get_ipython().system("python -m src.summarize")

## 7. A few translations (they must be clean Arabic that stops, not rambling text)

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_lora.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 2 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

import json, glob
for p in sorted(glob.glob("results/**/predictions.jsonl", recursive=True), key=os.path.getmtime)[-1:]:
    print(p)
    for line in open(p, encoding="utf-8").readlines()[:5]:
        r = json.loads(line)
        print("INPUT:", r["input"]); print("REF  :", r["reference"]); print("PRED :", repr(r["prediction"])); print("-" * 60)

## 8. Package the results (adapters included, checkpoints already deleted)

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_lora.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 2 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

import shutil
zname = f"results_session{SESSION}.zip"
get_ipython().system(f"zip -qr {zname} results")
shutil.copy(zname, f"/kaggle/working/{zname}")
get_ipython().system(f"ls -la /kaggle/working/{zname}")